### Fact Orders

In [0]:
df = spark.sql("select * from datapipeline.silver.orders")
df.show()

+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|flag|rank_flag|row_flag|
+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|   1|        1|       1|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   1|        1|       2|
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   1|        1|       3|
|  O01765|     C01515|     P0498|2023-05-11 00:00:00|       5|      9952.9|2023|   1|        1|       4|
|  O06790|     C01819|     P0498|2023-02-27 00:00:00|       5|      9952.9|2023|   1|        1|       5|
|  O07763|     C00471|     P0440|2023-04-12 00:00:00|       5|     9916.25|2023|   1|        6|       6|
|  O03989|     C00631|     P0440|2023-11-20 00:00:00|  

In [0]:
dim_cust = spark.sql("select DimCustomerKey, customer_id as dim_customer_id from datapipeline.gold.dimcustomers")

dim_prod =  df_dimpro = spark.sql("select product_id as DimProductKey, product_id as dim_product_id from datapipeline.gold.dimproduct")

Fact Dataframe

In [0]:
df_fact = df.join(dim_cust,df["customer_id"]==dim_cust["dim_customer_id"],how="left")\
    .join(df_dimpro,df["product_id"]==df_dimpro["dim_product_id"],how="left")

df_fact_new = df_fact.drop('dim_customer_id','dim_product_id','customer_id','product_id')
df_fact.show()


+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+--------------+---------------+-------------+--------------+
|order_id|customer_id|product_id|         order_date|quantity|total_amount|year|flag|rank_flag|row_flag|DimCustomerKey|dim_customer_id|DimProductKey|dim_product_id|
+--------+-----------+----------+-------------------+--------+------------+----+----+---------+--------+--------------+---------------+-------------+--------------+
|  O03502|     C00805|     P0498|2023-09-24 00:00:00|       5|      9952.9|2023|   1|        1|       1|          1300|         C00805|         NULL|          NULL|
|  O03660|     C01001|     P0498|2023-10-08 00:00:00|       5|      9952.9|2023|   1|        1|       2|          1937|         C01001|         NULL|          NULL|
|  O00957|     C01449|     P0498|2023-10-05 00:00:00|       5|      9952.9|2023|   1|        1|       3|           102|         C01449|         NULL|          NULL|
|  O01765|

In [0]:
from delta import *

In [0]:
if spark.catalog.tableExists("datapipeline.gold.FactOrders"):
    
    dlt_obj = DeltaTable.forName(spark, "datapipeline.gold.FactOrders")
    
    dlt_obj.alias("trg").merge(df_fact_new.alias("src"), "trg.order_id = src.order_id AND src.DimCustomerKey = trg.DimCustomerKey AND src.DimProductKey = trg.DimProductKey")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df_fact_new.write.format("delta")\
        .option("path","abfss://gold@strdatabricks.dfs.core.windows.net/FactOrders")\
        .saveAsTable("datapipeline.gold.FactOrders")

In [0]:
%sql select * from datapipeline.gold.factorders

order_id,order_date,quantity,total_amount,year,flag,rank_flag,row_flag,DimCustomerKey,DimProductKey
O03502,2023-09-24T00:00:00.000Z,5,9952.9,2023,1,1,1,1300,null
O03660,2023-10-08T00:00:00.000Z,5,9952.9,2023,1,1,2,1937,null
O00957,2023-10-05T00:00:00.000Z,5,9952.9,2023,1,1,3,102,null
O01765,2023-05-11T00:00:00.000Z,5,9952.9,2023,1,1,4,500,null
O06790,2023-02-27T00:00:00.000Z,5,9952.9,2023,1,1,5,637,null
O07763,2023-04-12T00:00:00.000Z,5,9916.25,2023,1,6,6,296,P0440
O03989,2023-11-20T00:00:00.000Z,5,9916.25,2023,1,6,7,940,P0440
O08261,2023-11-29T00:00:00.000Z,5,9858.85,2023,1,8,8,1807,P0165
O03746,2023-10-26T00:00:00.000Z,5,9858.85,2023,1,8,9,1851,P0165
O03272,2023-02-15T00:00:00.000Z,5,9858.85,2023,1,8,10,1630,P0165
